In [19]:
# Import Libraries
import pandas as pd
import numpy as np
import torch
from torch.optim import AdamW
from google.colab import files
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import os

!pip install transformers datasets torch

!pip install -q transformers datasets

In [3]:
# Load Data sets

train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test_features.csv")
sample_submission_df = pd.read_csv("/content/sample_submission.csv")

# Display a preview of the data
train_preview = train_df.head()
test_preview = test_df.head()
sample_submission_preview = sample_submission_df.head()

train_preview, test_preview, sample_submission_preview

(    ID                                               Text  Category
 0  969  @JuliaBradbury @SimonCalder @walsop @HodderPRI...         0
 1  241  or here https://t.co/R2tO79Easn … .An in house...         1
 2  820  @britshmuseum @thehistoryguy Gosh periscope is...         2
 3  693  @Ophiolatrist britishmuseum The stupid #French...         1
 4  421  @SassyClde We won't stop til @britishmuseum du...         1,
      ID                                               Text
 0  1861  Goodbye @kettlesyard see you in .25 years! htt...
 1   354  @BBC_Culture @PlymouthMuseum Oh dear, why not ...
 2  1334  Fantastic @johnmcdonnellMP standing up for wor...
 3   906  @BBC_Culture @PlymouthMuseum Oh dear, why not ...
 4  1290                @britishmuseum @TripAdvisor it is !,
      ID  Prediction
 0  1861           0
 1   354           0
 2  1334           0
 3   906           0
 4  1290           0)

In [16]:
###################################################################################################################################
# Model 1: BERT using Higging Face + PyTorch
###################################################################################################################################

# Split into train/validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['Text'], train_df['Category'], test_size=0.1, random_state=42
)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize training and validation data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_df['Text']), truncation=True, padding=True, max_length=128)

# Create Torch Dataset Class
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

# Create datasets
train_dataset = SentimentDataset(train_encodings, list(train_labels))
val_dataset = SentimentDataset(val_encodings, list(val_labels))
test_dataset = SentimentDataset(test_encodings)

# Load Pretrained BERT model
# Load model with 4 output labels (0-3)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

#Train the model
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Initialize optimizer from torch.optim
optim = AdamW(model.parameters(), lr=5e-5)

# Training loop
epochs = 3
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optim.step()
        optim.zero_grad()
        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())

# Evaluation on Validation data set
model.eval()
preds, true = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
        true.extend(batch['labels'].cpu().numpy())

print("Validation Accuracy:", accuracy_score(true, preds))

#Predict on test set
test_loader = DataLoader(test_dataset, batch_size=16)
test_preds = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())

#Prepare submission file
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Prediction': test_preds
})
submission.to_csv("submission1.csv", index=False)

# Download it
files.download("submission1.csv")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 2: 100%|██████████| 90/90 [00:19<00:00,  4.58it/s, loss=0.0717]


Validation Accuracy: 0.98125


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
###################################################################################################################################
# Model 2: DistilBERT for Sentiment Classification
###################################################################################################################################

#Prepare and tokenize
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['Text'], train_df['Category'], test_size=0.1, random_state=42
)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_df['Text']), truncation=True, padding=True, max_length=128)

#Create dataset class
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = SentimentDataset(train_encodings, list(train_labels))
val_dataset = SentimentDataset(val_encodings, list(val_labels))
test_dataset = SentimentDataset(test_encodings)

#Load DistillBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=4
)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

#Train the model
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 3
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

#Evaluation on validation dataset
model.eval()
val_preds, val_labels_true = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        val_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
        val_labels_true.extend(batch['labels'].cpu().numpy())

print("Validation Accuracy:", accuracy_score(val_labels_true, val_preds))

#Predict on test set
test_loader = DataLoader(test_dataset, batch_size=16)
test_preds = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())

#Create submission file
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Prediction': test_preds
})

submission.to_csv("submission2.csv", index=False)
files.download("submission2.csv")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 2: 100%|██████████| 90/90 [00:10<00:00,  8.44it/s, loss=0.00409]


Validation Accuracy: 0.98125


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
###################################################################################################################################
# Model 2: DistilBERT for Sentiment Classification
###################################################################################################################################
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# === TOKENIZATION === #
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_df['Text']), truncation=True, padding=True, max_length=128)

# === CUSTOM DATASET === #
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = SentimentDataset(train_encodings, list(train_labels))
val_dataset = SentimentDataset(val_encodings, list(val_labels))
test_dataset = SentimentDataset(test_encodings)

# === LOAD MODEL === #
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# === TRAINING SETUP === #
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
optimizer = AdamW(model.parameters(), lr=2e-5)

# === TRAIN LOOP === #
epochs = 3
for epoch in range(epochs):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loop.set_description(f"Epoch {epoch}")
        loop.set_postfix(loss=loss.item())

# === VALIDATION === #
model.eval()
val_preds, val_true = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        val_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
        val_true.extend(batch['labels'].cpu().numpy())

acc = accuracy_score(val_true, val_preds)
print(f"Validation Accuracy: {acc:.4f}")

# === PREDICT TEST SET === #
test_loader = DataLoader(test_dataset, batch_size=16)
test_preds = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        test_preds.extend(torch.argmax(logits, axis=1).cpu().numpy())

# === CREATE SUBMISSION FILE === #
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'Prediction': test_preds
})

submission.to_csv("submission3.csv", index=False)

# === DOWNLOAD SUBMISSION FILE === #
files.download("submission3.csv")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 2: 100%|██████████| 90/90 [00:23<00:00,  3.86it/s, loss=0.156]


Validation Accuracy: 0.9750


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>